# 1. Coverage matching

Load datasets, inspect metadata, compare MEG manipulations, assess channel count, fit PCA, and assess assignment stability.

`paired_coverage` is the existing code name for pairwise coverage. Subject associations are always random and reproducible. MEG z-scoring and iEEG ×1000 are preprocessing corrections. Optional block scaling divides each constructed dataset by the square root of its total sample variance, after condition averaging or stacking. It does not balance subjects independently.

Analysis cells print an executable cluster command and load compatible completed results. Set `RECOMPUTE=True` to request a new run; after it finishes, set it back to `False`. Adjust the root and data paths for the execution machine. Previous outputs are preserved. Cached results assume the source files have not been replaced in place.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
MEG_DIR = ROOT / 'MEG' / 'dataMEG'
IEEG_DIR = ROOT / 'ieeg_shortWOBS_fs250'
OUTPUT_DIR = ROOT / 'out' / 'coverage'
METADATA_CSV = None  # Optional metadata file; otherwise use src.setting.GetInfo.
PROJECT_PATH = None  # Optional override for GetInfo.
BLOCK_SCALING = True
N_COMPONENTS = 10
CONDITION_MODE = 'average'  # 'average' or 'stack'
CORRECT_IEEG_X1000 = True
CORRECT_MEG_ZSCORE = True
CONDITIONS = [1, 2]
SFREQ = 250.
MEG_TMIN = None  # None uses the iEEG epoch start; verify upstream alignment.
MEG_TIMES_FILE = None
SEED = 2026
CHANNEL_STEP = 100
CHANNEL_REPEATS = 20
STABILITY_REPEATS = 20
MAX_GRAM_GIB = 2.

sys.path[:0] = [str(ROOT), str(ROOT.parent), str(ROOT.parent / 'LB')]
from vs1_run_coverage import load_datasets, prepare_analysis, load_pca_results
from vs1_coverage_matching_utils import variance_summary, correlate_timecourses, correlate_weights, compare_pca_to_ieeg
from vs1_coverage_information import load_information, COMPARISONS
from vs1_elect_number_effect import load_elect_number_effect
from vs1_coverage_stability import load_stability_results
from vs1_coverage_figures import apply_style, MODALITY_COLORS, SETUP_COLORS, SETUP_LINESTYLES, show_coverage, plot_pca_weights
apply_style()


## 1. Load the datasets

In [ ]:
INPUT_DIRS = {'meg': MEG_DIR, 'ieeg': IEEG_DIR}
CONFIG_OUTPUT_DIR = OUTPUT_DIR

CONFIG = dict(
    data=dict(meg_dir=str(INPUT_DIRS['meg']), ieeg_dir=str(INPUT_DIRS['ieeg']),
              metadata_csv=str(METADATA_CSV) if METADATA_CSV else None,
              project_path=str(PROJECT_PATH) if PROJECT_PATH else None,
              meg_coordinate_unit='m', ieeg_coordinate_unit='mm',
              meg_scaling='channel_zscore' if CORRECT_MEG_ZSCORE else 'none',
              ieeg_scaling='none', ieeg_multiplier=1000. if CORRECT_IEEG_X1000 else 1.,
              meg_tmin=MEG_TMIN, meg_times_file=str(MEG_TIMES_FILE) if MEG_TIMES_FILE else None,
              sfreq=SFREQ, conditions=CONDITIONS),
    output_dir=str(CONFIG_OUTPUT_DIR), block_scaling=BLOCK_SCALING, condition_mode=CONDITION_MODE,
    n_components=N_COMPONENTS, seed=SEED, channel_step=CHANNEL_STEP,
    channel_repeats=CHANNEL_REPEATS, stability_repeats=STABILITY_REPEATS,
    max_gram_gib=MAX_GRAM_GIB)
datasets = load_datasets(CONFIG)
ieeg = datasets['iEEG']


## 2. Metadata, variance, feature counts, and brain coverage

Subject counts refer to contributing recordings, including those averaged together. MEG features are sources, not electrodes. Unique coordinates quantify sampled locations, not anatomical volume. Coverage maps display at most `MAX_COVERAGE_POINTS` evenly selected unique locations. Variances reflect the configured corrections and scaling; unscaled MEG and iEEG variances are not directly comparable in physical units.


In [ ]:
MAX_COVERAGE_POINTS = 20000

summary = variance_summary(datasets)
for name, data in datasets.items():
    n_subjects = (len(data.source_data['ieeg_subjects']) if name == 'iEEG'
                  else len(data.pairing) if name in ('paired_coverage', 'random_control')
                  else len(data.source_data['meg_subjects']))
    summary.loc[name, 'n_subjects'] = n_subjects
    summary.loc[name, 'n_unique_locations'] = len(data.metadata[['x', 'y', 'z']].drop_duplicates())
    summary.loc[name, 'scale'] = data.scale
    summary.loc[name, 'feature_type'] = 'electrode' if name == 'iEEG' else 'source'
display(summary)
display(ieeg.source_data['trial_counts'])
display(ieeg.metadata)
for name in ('paired_coverage', 'random_control'):
    display(datasets[name].matching)


In [ ]:
FIGSIZE = (13, 4)

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, constrained_layout=True)
colors = [MODALITY_COLORS['iEEG'] if name == 'iEEG' else SETUP_COLORS[name] for name in summary.index]
for ax, column in zip(axes, ['n_subjects', 'n_features', 'total_variance']):
    ax.bar(summary.index, summary[column], color=colors)
    ax.set(ylabel=column); ax.tick_params(axis='x', labelrotation=70); ax.grid(True)
plt.show()
for data in datasets.values():
    show_coverage(data, max_points=MAX_COVERAGE_POINTS)


## 3. Direct MEG comparisons by manipulation

Compare participant averaging, coverage restriction in averaged data, and coverage plus participant contributions. Report symmetric temporal overlap, both directions of retained-PC variance capture, normalized spectra, and full-spectrum participation ratio. No construction is a ground truth. All comparisons are descriptive and in sample.


In [ ]:
INPUT_DIR = OUTPUT_DIR
CONFIG_FILE = INPUT_DIR / 'config.json'
RECOMPUTE = False

result_dir = prepare_analysis(dict(CONFIG, output_dir=str(INPUT_DIR)), 'manipulations', CONFIG_FILE, recompute=RECOMPUTE)
meg_information = load_information(result_dir) if result_dir is not None else None


In [ ]:
FIGSIZE = (16, 11)

if meg_information is not None:
    display(meg_information['summary'])
    display(meg_information['metrics'])
    fig, axes = plt.subplots(3, 3, figsize=FIGSIZE, constrained_layout=True)
    for col, (manipulation, a, b, title) in enumerate(COMPARISONS):
        rows = meg_information['metrics'].query('manipulation == @manipulation')
        for metric, style in [('temporal_overlap', '-'), ('A_variance_captured_by_B', '--'), ('B_variance_captured_by_A', ':')]:
            axes[0, col].plot(rows.k, rows[metric], linestyle=style, color=MODALITY_COLORS['MEG'], label=metric)
        axes[0, col].set(title=title, xlabel='Number of PCs', ylabel='Fraction', ylim=(0, 1.02))
        axes[0, col].legend(fontsize=7)
        for name in (a, b):
            spectrum = meg_information['spectra'].query('dataset == @name and component <= @N_COMPONENTS')
            axes[1, col].plot(spectrum.component, spectrum.cumulative_variance, color=SETUP_COLORS[name], linestyle=SETUP_LINESTYLES[name], label=name)
        axes[1, col].set(xlabel='Number of PCs', ylabel='Cumulative explained variance', ylim=(0, 1.02))
        axes[1, col].legend(fontsize=7)
        values = meg_information['summary'].set_index('dataset').loc[[a, b], 'effective_dimensionality']
        axes[2, col].bar(['A: '+a, 'B: '+b], values, color=[SETUP_COLORS[a], SETUP_COLORS[b]])
        axes[2, col].set(ylabel='Participation ratio'); axes[2, col].tick_params(axis='x', labelrotation=20)
    for ax in axes.flat: ax.grid(True)
    plt.show()


## 4. MEG channel-count effect

Nested random channel subsets of full-concatenated MEG, up to 20,000 channels, compared with the fixed iEEG reference. The global component count applies throughout. Channels are sampled across all subjects without subject balancing. With block scaling enabled, each selected subset is normalized independently; its total variance is therefore one.


In [ ]:
INPUT_DIR = OUTPUT_DIR
CONFIG_FILE = INPUT_DIR / 'config.json'
RECOMPUTE = False

result_dir = prepare_analysis(dict(CONFIG, output_dir=str(INPUT_DIR)), 'channels', CONFIG_FILE, recompute=RECOMPUTE)
channel_effect = load_elect_number_effect(result_dir) if result_dir is not None else None


In [ ]:
MIN_CHANNELS = None
MAX_CHANNELS = None
FIGSIZE = (14, 4)

if channel_effect is not None:
    rows = channel_effect['metrics']
    if MIN_CHANNELS is not None: rows = rows[rows.n_channels >= MIN_CHANNELS]
    if MAX_CHANNELS is not None: rows = rows[rows.n_channels <= MAX_CHANNELS]
    display(rows)
    fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, constrained_layout=True)
    for ax, metric in zip(axes, ['variance_fraction_retained', 'subspace_overlap', 'matched_abs_r']):
        values = rows.groupby('n_channels')[metric].agg(['median', 'min', 'max'])
        ax.plot(values.index, values['median'], color=MODALITY_COLORS['MEG'])
        ax.fill_between(values.index, values['min'], values['max'], color=MODALITY_COLORS['MEG'], alpha=.2)
        ax.set(xlabel='MEG source count', ylabel=metric, ylim=(0, 1.02)); ax.grid(True)
    plt.show()


## 5. PCA for all datasets

Display component time courses, explained variance, absolute spatial weights, and correlations with iEEG. Signed Spearman correlations compare all component pairs without sign alignment. Weight comparisons use electrode-slot correspondence; random-control slots do not imply anatomical correspondence. Temporal subspace comparisons are descriptive and in sample.


In [ ]:
INPUT_DIR = OUTPUT_DIR
CONFIG_FILE = INPUT_DIR / 'config.json'
RECOMPUTE = False

result_dir = prepare_analysis(dict(CONFIG, output_dir=str(INPUT_DIR)), 'pca', CONFIG_FILE, recompute=RECOMPUTE)
pca = load_pca_results(result_dir, datasets) if result_dir is not None else None


In [ ]:
PLOT_COMPONENTS = N_COMPONENTS
BRAIN_COMPONENTS = min(3, N_COMPONENTS)
BRAIN_PLOT_TYPE = 'glass'
FIGSIZE = (13, 3)

if pca is not None:
    times = ieeg.source_data['times']
    for name, result in pca.items():
        color = MODALITY_COLORS['iEEG'] if name == 'iEEG' else SETUP_COLORS[name]
        display(pd.DataFrame({'PC': np.arange(1, len(result.explained_variance)+1),
                              'variance': result.explained_variance,
                              'variance_fraction': result.explained_variance_ratio}))
        k = min(PLOT_COMPONENTS, result.scores.shape[1])
        fig, axes = plt.subplots(k, 1, figsize=(FIGSIZE[0], FIGSIZE[1]*k), squeeze=False, constrained_layout=True)
        for pc, ax in enumerate(axes[:, 0]):
            traces = result.scores[:, pc].reshape(-1, len(times))
            for condition, trace in enumerate(traces):
                ax.plot(times, trace, color=color, linestyle=['-', '--', ':', '-.'][condition % 4],
                        label='Condition average' if CONDITION_MODE == 'average' else str(CONDITIONS[condition]))
            ax.set(title=f'{name}: PC{pc+1}', xlabel='Time (s)', ylabel='Score'); ax.legend(); ax.grid(True)
        plt.show()
        fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
        ax.plot(np.arange(1, len(result.explained_variance_ratio)+1), result.explained_variance_ratio, 'o-', color=color)
        ax.set(title=name, xlabel='PC', ylabel='Explained variance fraction'); ax.grid(True)
        plt.show()
        plot_pca_weights(result, n_components=BRAIN_COMPONENTS, plot_type=BRAIN_PLOT_TYPE, absolute=True)


In [ ]:
CORRELATION_CMAP = 'RdBu_r'
FIGSIZE = (12, 5)

if pca is not None:
    time_correlations, weight_correlations = {}, {}
    for name in datasets:
        if name == 'iEEG': continue
        time_correlations[name] = correlate_timecourses(pca[name], pca['iEEG'], plot=False)
        weight_correlations[name] = correlate_weights(pca[name], pca['iEEG'], plot=False)
    comparison, component_pairs = compare_pca_to_ieeg(pca, n_components=range(1, N_COMPONENTS+1), plot=False)
    display(comparison)
    for name in time_correlations:
        fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, constrained_layout=True)
        for ax, title, matrix in zip(axes, ['Time courses', 'Weights in electrode-slot order'],
                                     [time_correlations[name], weight_correlations[name]]):
            sns.heatmap(matrix, ax=ax, cmap=CORRELATION_CMAP, vmin=-1, vmax=1,
                        linewidths=.5, linecolor='white', cbar_kws={'label': 'Signed Spearman r'})
            ax.set(title=f'{name}: {title}'); ax.grid(True)
        plt.show()


## 6. Stability under random subject reassignment

Keep iEEG, both participant cohorts, and electrode-slot order fixed. Shuffle MEG-to-iEEG subject assignments. For `random_control`, fix one source-index permutation per MEG subject across repetitions, avoiding additional random-source variability.

Compare time courses and weights against the baseline, between shuffled assignments, and against fixed iEEG. Each domain uses its own one-to-one component assignment maximizing absolute Pearson correlation. Weight correlations therefore describe electrode-slot stability, not stability at identical MEG source locations. Pairwise comparisons share repetitions and are not independent observations. Component matching resolves signs and ordering, but not rotations of nearly degenerate PCA subspaces.


In [ ]:
INPUT_DIR = OUTPUT_DIR
CONFIG_FILE = INPUT_DIR / 'config.json'
RECOMPUTE = False

result_dir = prepare_analysis(dict(CONFIG, output_dir=str(INPUT_DIR)), 'stability', CONFIG_FILE, recompute=RECOMPUTE)
stability = load_stability_results(result_dir, expected_experiment='pairing') if result_dir is not None else None


In [ ]:
FIGSIZE = (16, 8)

if stability is not None:
    display(stability['paired_control_differences'])
    fig, axes = plt.subplots(2, 3, figsize=FIGSIZE, constrained_layout=True)
    for row, domain in enumerate(['timecourses', 'weights']):
        for col, comparison in enumerate(['to_baseline_pairing', 'between_pairings', 'cross_modal']):
            ax = axes[row, col]
            for index, name in enumerate(['paired_coverage', 'random_control']):
                rows = stability['metrics'].query('domain == @domain and comparison == @comparison and dataset == @name and repeat >= 0 and status == "ok"')
                if len(rows):
                    ax.boxplot([rows.correlation.to_numpy()], positions=[index], widths=.4, showfliers=False)
                    ax.scatter(index+np.linspace(-.12, .12, len(rows)), rows.correlation,
                               color=SETUP_COLORS[name], alpha=.4, s=12)
                if comparison == 'cross_modal':
                    baseline = stability['metrics'].query('domain == @domain and comparison == @comparison and dataset == @name and repeat == -1')
                    ax.scatter([index], baseline.correlation, marker='*', color=MODALITY_COLORS['iEEG'], s=100)
            ax.set(title=f'{domain}: {comparison}', xticks=[0, 1], xticklabels=['Paired coverage', 'Random control'],
                   ylabel='Mean matched |Pearson r|', ylim=(-.02, 1.02)); ax.grid(True)
    plt.show()
